In [ ]:
import pandas as pd
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)

url = "https://cidbregistration.govmu.org/cidbfront/pages/listOfValidCertificates.xhtml"

driver.get(url)

time.sleep(5)

total_pages = 70

for pg_no in range(1, total_pages + 1):

    print(f"\n========== SCRAPING PAGE {pg_no} ==========")

    try:
        # PAGE CHANGE ONLY IF NOT PAGE 1
        if pg_no > 1:

            page_btn = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        f"//a[normalize-space()='{pg_no}']"
                    )
                )
            )

            driver.execute_script(
                "arguments[0].click();",
                page_btn
            )

            time.sleep(4)

    except Exception as e:
        print(f"Pagination Error on Page {pg_no}: {e}")
        continue

    soup = BeautifulSoup(driver.page_source, "html.parser")

    table = soup.find("table")

    if not table:
        print(f"Table not found on page {pg_no}")
        continue

    # HEADERS
    headers = []

    for th in table.find_all("th"):
        txt = th.get_text(strip=True)
        if txt:
            headers.append(txt)

    print("HEADERS:", headers)

    # ROWS
    rows_data = []

    for row in table.find_all("tr"):

        cols = row.find_all("td")

        if not cols:
            continue

        row_dict = {}

        for i, header in enumerate(headers):

            if i < len(cols):
                row_dict[header] = cols[i].get_text(strip=True)
            else:
                row_dict[header] = ""

        rows_data.append(row_dict)

    df = pd.DataFrame(rows_data)

    print(f"Rows Found: {len(df)}")

    # ✅ ONLY CSV SAVE
    csv_name = f"cidb_page_{pg_no}.csv"
    df.to_csv(csv_name, index=False)

    print(f"Saved CSV: {csv_name}")
    print(f"PAGE {pg_no} COMPLETED")

driver.quit()

print("\nALL PAGES SCRAPED SUCCESSFULLY 🎉")

In [ ]:
# 